[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C05_Safety_Evals_Course/07_safety_cases/07_safety_cases.ipynb)

# 07 · Safety Case 与治理报告 <span style="font-size:0.6em">MODULE 07 / 8 · 纯 CPU</span>

把前六个模块产出的评测证据，组装成一份**可计算、可审查**的 safety case
[Clymer 2024, arXiv:2403.10462; Buhl 2024, arXiv:2410.21572]。

本 notebook 全程纯 Python 标准库（matplotlib 可选），实现一个最小但完整的 safety case 工具链：

1. **数据结构**：`Claim`（主张）与 `Evidence`（证据）组成的树
2. **示例 case**：「模型 M 在部署 D 下自主性风险可接受」→ 3 个子主张 → 7 条合成证据
3. **置信度传播**：叶子 → 根（`min` 与 `noisy-AND` 两种语义对比）
4. **ASCII 渲染器**：把树打印成带置信度与状态符号的缩进文本
5. **敏感性分析**：tornado 图找 load-bearing 证据
6. **共因失效**：共享假设崩塌 vs 独立失效的对比
7. **报告生成**：输出 system-card 风格 markdown 章节
8. 3 道 ✏️ 练习 + 📖 参考答案 + 🎓 全课收尾

> ⚠️ 防御/治理视角：所有证据均为**合成数据**，仅用于演示论证方法论。

## 1 · 数据结构

- `Evidence`：来源 / 类型 / 置信度（0–1，"该证据正确支撑其主张的概率"）/ 日期 /
  隐含假设列表（共因失效分析用）/ 备注
- `Claim`：文本 + 子主张列表 + 直接挂载的证据列表


In [ ]:
from dataclasses import dataclass, field
import math, copy

@dataclass
class Evidence:
    eid: str
    source: str          # 来源（哪个模块/哪次评测产出）
    etype: str           # 类型：capability_eval / redteam / control_eval / monitoring / ...
    confidence: float    # 0-1：该证据正确支撑其叶子主张的概率
    date: str            # 生成日期（证据会过期）
    assumptions: tuple = ()   # 隐含假设 id（共因失效分析用）
    note: str = ''       # 具体数字/上下文，进报告用

@dataclass
class Claim:
    cid: str
    text: str
    subclaims: list = field(default_factory=list)
    evidence: list = field(default_factory=list)

def iter_evidence(claim):
    # 深度优先遍历整棵树的所有证据
    for ev in claim.evidence:
        yield ev
    for sub in claim.subclaims:
        yield from iter_evidence(sub)

def find_evidence(claim, eid):
    for ev in iter_evidence(claim):
        if ev.eid == eid:
            return ev
    return None

print('数据结构就绪：Evidence / Claim / iter_evidence / find_evidence')


## 2 · 构建示例 case

顶层主张：**「模型 M 在部署 D 下自主性（autonomy/ARA）风险可接受」**，按"能力—护栏—监控"三道防线分解：

- **C1 能力未达阈值**（inability 论证腿）← 模块 02/05 的产出
- **C2 部署护栏有效**（control 论证腿）← 模块 03/06 的产出
- **C3 监控覆盖且可回滚** ← 模块 06 的产出

注意 E1 / E2 / E4 都带着同一个隐含假设 `elicitation_sufficient`（"我们已把模型真实水平逼出来了"，模块 05 的主题）——第 6 节它会塌给你看。

## 3 · 置信度传播：叶子 → 根

聚合语义（与 HTML 讲解 §4 一致）：

- **叶子主张**：多条独立证据互相佐证，**noisy-OR** 叠加 $c(\mathrm{leaf}) = 1-\prod_i(1-c_i)$；无证据 → 0
- **内部主张**：所有子主张须同时成立（合取）。两种语义：
  - `min`（weakest-link）：$c = \min_j c_j$ —— 短板决定论，保守且对链长不敏感
  - `noisy_and`（独立联合）：$c = \prod_j c_j$ —— 论证链越长越脆，如实反映合取代价
- 若内部节点还挂有直接证据，把这些证据的 noisy-OR 结果当作一个额外 child


In [ ]:
def build_case():
    c1 = Claim('C1', '能力未达阈值：模型在自主性危险能力评估上显著低于触发线', evidence=[
        Evidence('E1', '模块02 · 危险能力评估', 'capability_eval', 0.85, '2026-05-12',
                 ('elicitation_sufficient',), '自主复制任务套件得分 12% (95% CI [8,16])，阈值 40%'),
        Evidence('E2', '模块02 · 长任务评估(METR 式)', 'capability_eval', 0.75, '2026-05-14',
                 ('elicitation_sufficient',), '50%-task time horizon 远低于自主性门槛'),
        Evidence('E3', '模块05 · 完整性审计', 'integrity_audit', 0.60, '2026-05-20',
                 (), '微调+工具增强 elicitation 后分数仍低；未检出 sandbagging 信号'),
    ])
    c2 = Claim('C2', '护栏有效：即使能力被低估，部署协议可拦截危险动作', evidence=[
        Evidence('E4', '模块06 · control 评测', 'control_eval', 0.80, '2026-05-22',
                 ('elicitation_sufficient',), 'trusted monitoring 协议 safety 估计 92%（red team 攻击下）'),
        Evidence('E5', '模块03 · 红队 campaign', 'redteam', 0.70, '2026-05-18',
                 (), '对护栏的 3 轮人工+自动红队，覆盖度报告达标，未发现绕过'),
    ])
    c3 = Claim('C3', '监控覆盖：部署后行为被持续监控且可快速回滚', evidence=[
        Evidence('E6', '模块06 · 监控评测', 'monitoring', 0.75, '2026-05-25',
                 (), '行为/CoT 监控器 AUROC 0.94，2% 审计预算下捕获率达标'),
        Evidence('E7', '运维 · 事故响应', 'ops_drill', 0.65, '2026-05-28',
                 (), '审计抽样 + 人工 escalation 演练：回滚 SLA 达标'),
    ])
    return Claim('C0', '模型 M 在部署 D 下自主性风险可接受', subclaims=[c1, c2, c3])

case = build_case()
n_ev = len(list(iter_evidence(case)))
print(f'示例 case：1 个顶层主张 / {len(case.subclaims)} 个子主张 / {n_ev} 条证据')


In [ ]:
def propagate(claim, mode='noisy_and'):
    # 叶子: 证据 noisy-OR；内部: 子主张(+直接证据的 noisy-OR)做 min 或乘积
    children = [propagate(s, mode) for s in claim.subclaims]
    if claim.evidence:
        miss = 1.0
        for ev in claim.evidence:
            miss *= (1.0 - ev.confidence)
        children.append(1.0 - miss)
    if not children:
        return 0.0
    return min(children) if mode == 'min' else math.prod(children)

for sub in case.subclaims:
    print(f'{sub.cid}  noisy-OR 聚合后置信度 = {propagate(sub):.4f}   ({sub.text[:14]}...)')
print('-' * 56)
for mode in ('min', 'noisy_and'):
    print(f'根置信度  mode={mode:<9} : {propagate(case, mode):.4f}')
# 解读：min=0.9125 只看最短板 C3；noisy_and=0.845 反映三道防线必须同时成立的联合代价。
# 哪个对？都不"对"——它们是论证强度的乐观/悲观两个端点，汇报时应同时给出。


## 4 · ASCII 渲染器

把树打印成审查者可读的缩进文本。状态符号：`●` ≥0.8 ／ `◐` 0.5–0.8 ／ `○` <0.5；
`⚑` 标记证据携带的隐含假设。

## 5 · 敏感性分析：找 load-bearing 证据

逐条证据把置信度扰动 $\pm 0.2$（截断到 $[0,1]$），用 `noisy_and` 重算根置信度，画 **tornado 图**。
摆动幅度最大的证据就是 <b>load-bearing evidence</b>——整个 case 实际压在它身上：复审资源优先投给它，它过期最伤。

> 预判一下再跑：7 条证据里哪条最承重？答案有点反直觉——不是置信度最低的 E3。


In [ ]:
def status_symbol(c):
    return '●' if c >= 0.8 else ('◐' if c >= 0.5 else '○')

def render_tree(claim, mode='noisy_and', indent=0):
    conf = propagate(claim, mode)
    pad = '    ' * indent
    print(f'{pad}{status_symbol(conf)} [{conf:.3f}] {claim.cid} — {claim.text}')
    for ev in claim.evidence:
        flag = '  ⚑' + ','.join(ev.assumptions) if ev.assumptions else ''
        print(f'{pad}    └─ {status_symbol(ev.confidence)} [{ev.confidence:.2f}] '
              f'{ev.eid} · {ev.etype} · {ev.source} · {ev.date}{flag}')
    for sub in claim.subclaims:
        render_tree(sub, mode, indent + 1)

render_tree(case)


In [ ]:
DELTA = 0.2
base = propagate(case, 'noisy_and')
rows = []
for ev in iter_evidence(case):
    orig = ev.confidence
    ev.confidence = max(0.0, orig - DELTA); lo = propagate(case, 'noisy_and')
    ev.confidence = min(1.0, orig + DELTA); hi = propagate(case, 'noisy_and')
    ev.confidence = orig                                   # 恢复
    rows.append((ev.eid, lo, hi))
rows.sort(key=lambda r: -(r[2] - r[1]))

print(f'基线根置信度 (noisy-AND) = {base:.4f}    扰动 = ±{DELTA}\n')
print('eid   低侧     tornado（│=基线）            高侧    摆动幅度')
for eid, lo, hi in rows:
    left  = int(round((base - lo) * 150))
    right = int(round((hi - base) * 150))
    bar = ' ' * (16 - left) + '◄' * left + '│' + '►' * right + ' ' * (16 - right)
    print(f'{eid:<4} {lo:.4f} {bar} {hi:.4f}   {hi-lo:.4f}')
print(f'\n→ load-bearing 证据：{rows[0][0]}。'
      '不是置信度最低的 E3（它有 E1/E2 佐证，noisy-OR 下被稀释），'
      '而是支撑面最薄的子主张里斜率最大的那条。')
try:
    import matplotlib.pyplot as plt
    eids = [r[0] for r in rows][::-1]
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.barh(eids, [r[2]-r[1] for r in rows][::-1],
            left=[r[1] for r in rows][::-1], color='#7aa6d9')
    ax.axvline(base, color='crimson', ls='--', lw=1, label=f'baseline {base:.3f}')
    ax.set_xlabel('root confidence (noisy-AND)'); ax.set_title(f'Tornado: evidence ±{DELTA}')
    ax.legend(); plt.tight_layout(); plt.show()
except Exception as e:
    print(f'(matplotlib 不可用，已用 ASCII tornado：{e})')


## 6 · 共因失效演示

E1 / E2 / E4 共享假设 `elicitation_sufficient`。noisy-OR 的叠加增益完全依赖证据独立；
若该假设崩塌（如发现模型 sandbagging，或社区出现强得多的 elicitation 技术——模块 05），
三条证据**同时**大幅降权（×0.1），而不是一次倒一条。

对比两种情形：**共因失效**（三条同时塌）vs **独立失效**（同样的降权一次只打击一条）。

## 7 · 治理报告生成

把 case 输出为 system-card 风格的 markdown 章节：主张 / 证据清单 / 局限与共因假设 / 复审触发条件。
对内完整版含全部脆弱点；对外披露版按 HTML §6 的分层原则脱敏（此处生成对内版）。


In [ ]:
ASSUMPTION, FACTOR = 'elicitation_sufficient', 0.1
shared = [ev.eid for ev in iter_evidence(case) if ASSUMPTION in ev.assumptions]
print(f'共享假设 {ASSUMPTION} 的证据: {shared}\n基线根置信度: {base:.4f}\n')

# 情形 A：共因失效——共享假设崩塌，三条证据同时 ×FACTOR
cc = copy.deepcopy(case)
for ev in iter_evidence(cc):
    if ASSUMPTION in ev.assumptions:
        ev.confidence *= FACTOR
root_cc = propagate(cc, 'noisy_and')

# 情形 B：独立失效——同样的 ×FACTOR 一次只打击一条
print('情形 B（独立失效，每次只倒一条）:')
worst = 1.0
for eid in shared:
    one = copy.deepcopy(case)
    find_evidence(one, eid).confidence *= FACTOR
    r = propagate(one, 'noisy_and')
    worst = min(worst, r)
    print(f'  仅 {eid} 失效 → 根置信度 {r:.4f}')
print(f'\n情形 A（共因失效，{len(shared)} 条同时倒）→ 根置信度 {root_cc:.4f}')
print(f'\n对比：独立失效最坏 {worst:.4f}，共因失效 {root_cc:.4f} —— '
      '表面上有 7 条证据，实际一个假设就能砍掉一半论证强度。')
render_tree(cc)


In [ ]:
from collections import Counter

def to_markdown(case):
    lines = ['## 安全论证摘要（system card 草稿 · 对内版）', '']
    lines.append(f'**顶层主张**：{case.text}')
    lines.append(f'**根置信度**：min 语义 {propagate(case, "min"):.2f} ／ '
                 f'noisy-AND 语义 {propagate(case, "noisy_and"):.2f}')
    lines.append('')
    lines.append('### 子主张与证据')
    for sub in case.subclaims:
        lines.append(f'- **{sub.cid}**（聚合置信度 {propagate(sub):.2f}）：{sub.text}')
        for ev in sub.evidence:
            lines.append(f'    - `{ev.eid}` [{ev.confidence:.2f}, {ev.date}] '
                         f'{ev.source}（{ev.etype}）— {ev.note}')
    lines.append('')
    lines.append('### 局限与共因假设')
    counts = Counter(a for ev in iter_evidence(case) for a in ev.assumptions)
    for a, n in counts.items():
        if n >= 2:
            lines.append(f'- 假设 `{a}` 被 {n} 条证据共享 —— 共因失效风险，'
                         '其有效性须独立论证（见模块 05）')
    lines.append('- 置信度为评估者主观评级，未经校准研究（见 HTML §8 前沿）')
    lines.append('')
    lines.append('### 复审触发条件')
    for t in ['模型权重更新 / 继续训练', '工具接入或自主权限（affordance）变更',
              '社区出现新 elicitation 技术（inability 证据整体贬值）',
              '部署后监控告警越限或安全事故', '任一证据生成日期超过 90 天']:
        lines.append(f'- {t}')
    return '\n'.join(lines)

report = to_markdown(case)
print(report)


## ✏️ 练习 1：实现 `propagate_confidence(claim, mode)`

不要回看第 3 节的实现，自己重写一遍置信度传播（10–20 行）。规则：

1. **叶子**（无 `subclaims`）：对证据做 noisy-OR $1-\prod_i(1-c_i)$；无证据返回 `0.0`
2. **内部节点**：children = 各 subclaim 的传播值；**若还挂有直接证据**，把其 noisy-OR 也append 为一个 child（自测树的根节点就有直接证据，这是与演示树不同的地方）
3. `mode='min'` 取 `min(children)`；`mode='noisy_and'` 取乘积

提示：递归 + `math.prod`。先手算自测树：S1 = $1-0.2\times0.5=0.9$，S2 = $0.6$，根的 children = $[0.9, 0.6, 0.5]$。


In [ ]:
def propagate_confidence(claim, mode='noisy_and'):
    # TODO: 1) 递归计算各 subclaim 的传播值
    # TODO: 2) 若 claim.evidence 非空，append 其 noisy-OR
    # TODO: 3) 无 children 返回 0.0；否则按 mode 取 min 或乘积
    raise NotImplementedError


In [ ]:
# —— 自测（手算见练习说明）——
_s1 = Claim('S1', '子主张1', evidence=[Evidence('e1', 't', 't', 0.8, '2026-05-01'),
                                       Evidence('e2', 't', 't', 0.5, '2026-05-01')])
_s2 = Claim('S2', '子主张2', evidence=[Evidence('e3', 't', 't', 0.6, '2026-05-01')])
_rt = Claim('R', '根（带直接证据）', subclaims=[_s1, _s2],
            evidence=[Evidence('e0', 't', 't', 0.5, '2026-05-01')])

assert abs(propagate_confidence(_s1, 'noisy_and') - 0.9) < 1e-9   # 叶子 noisy-OR
assert abs(propagate_confidence(_s1, 'min') - 0.9) < 1e-9         # 叶子与 mode 无关
assert abs(propagate_confidence(_rt, 'min') - 0.5) < 1e-9         # min(0.9, 0.6, 0.5)
assert abs(propagate_confidence(_rt, 'noisy_and') - 0.27) < 1e-9  # 0.9*0.6*0.5
assert propagate_confidence(Claim('L', '空叶子'), 'min') == 0.0    # 边界：无证据
# 与演示实现在示例 case 上一致
assert abs(propagate_confidence(case, 'noisy_and') - propagate(case, 'noisy_and')) < 1e-12
print('✅ 练习 1 通过')


## ✏️ 练习 2：实现 `load_bearing(case, delta)`

返回按敏感度降序的 `[(eid, sensitivity), ...]`：

- 基线 = `propagate(case, 'noisy_and')`（可直接用第 3 节的 `propagate`）
- 对每条证据：把置信度临时下调 `delta`（用 `max(0.0, c - delta)` 截断），重算根置信度，
  `sensitivity = 基线 - 扰动后`，然后**恢复原值**（不得永久改动 case）
- 按 sensitivity 从大到小排序返回

提示：`iter_evidence` 遍历 + 临时修改/恢复，约 10 行。自测树手算：
A = $1-0.1\times0.2=0.98$，B = $0.6$，基线 = $0.588$；`b1` 降到 0.4 → 根 = $0.392$，敏感度 $0.196$。


In [ ]:
def load_bearing(case, delta=0.2):
    # TODO: 1) 计算基线根置信度（noisy_and）
    # TODO: 2) 逐条证据：临时下调 delta（截断到 >=0）→ 重算 → 记录差值 → 恢复
    # TODO: 3) 按敏感度降序返回 [(eid, sensitivity), ...]
    raise NotImplementedError


In [ ]:
# —— 自测 ——
_A = Claim('A', '强支撑', evidence=[Evidence('a1', 't', 't', 0.9, '2026-05-01'),
                                    Evidence('a2', 't', 't', 0.8, '2026-05-01')])
_B = Claim('B', '单证据弱支撑', evidence=[Evidence('b1', 't', 't', 0.6, '2026-05-01')])
_case2 = Claim('R2', '根', subclaims=[_A, _B])

out = load_bearing(_case2, delta=0.2)
assert len(out) == 3
assert out[0][0] == 'b1'                                  # 单证据子主张最承重
assert abs(out[0][1] - 0.196) < 1e-9                      # 手算：0.588 - 0.98*0.4
assert all(out[i][1] >= out[i+1][1] for i in range(len(out)-1))   # 降序
assert find_evidence(_case2, 'b1').confidence == 0.6      # 原树未被改动
# 边界：delta 大于置信度时截断到 0
out2 = load_bearing(_case2, delta=1.0)
assert abs(out2[0][1] - 0.588) < 1e-9                     # b1→0 时根=0
print('✅ 练习 2 通过')


## ✏️ 练习 3：实现 `common_cause_discount(case, assumption_id, factor)`

共因失效建模：返回一个**新的** case（`copy.deepcopy`，不改原树），其中所有
`assumptions` 含 `assumption_id` 的证据 `confidence *= factor`，其余证据不变。

提示：deepcopy + `iter_evidence` 过滤，约 5 行。自测树手算（factor=0.5）：
`a1` $0.8\to0.4$、`b1` $0.6\to0.3$、`a2` 不变；根（noisy-AND）从 $(1-0.2\times0.5)\times0.6=0.54$
降到 $(1-0.6\times0.5)\times0.3=0.21$。


In [ ]:
def common_cause_discount(case, assumption_id, factor):
    # TODO: 1) deepcopy 出新 case
    # TODO: 2) 遍历新 case 的证据，含 assumption_id 的 confidence *= factor
    # TODO: 3) 返回新 case（原 case 不得被修改）
    raise NotImplementedError


In [ ]:
# —— 自测 ——
_A3 = Claim('A', 'a', evidence=[Evidence('a1', 't', 't', 0.8, '2026-05-01', ('ELIC',)),
                                Evidence('a2', 't', 't', 0.5, '2026-05-01')])
_B3 = Claim('B', 'b', evidence=[Evidence('b1', 't', 't', 0.6, '2026-05-01', ('ELIC',))])
_case3 = Claim('R3', '根', subclaims=[_A3, _B3])

new3 = common_cause_discount(_case3, 'ELIC', 0.5)
assert abs(find_evidence(new3, 'a1').confidence - 0.4) < 1e-9   # 共享假设：降权
assert abs(find_evidence(new3, 'b1').confidence - 0.3) < 1e-9   # 共享假设：降权
assert find_evidence(new3, 'a2').confidence == 0.5              # 不共享：不动
assert find_evidence(_case3, 'a1').confidence == 0.8            # 原树未被修改
old_r, new_r = propagate(_case3, 'noisy_and'), propagate(new3, 'noisy_and')
assert abs(old_r - 0.54) < 1e-9 and abs(new_r - 0.21) < 1e-9 and new_r < old_r
# 边界：无证据共享该假设时，根置信度不变
same = common_cause_discount(_case3, 'NO_SUCH', 0.5)
assert abs(propagate(same, 'noisy_and') - old_r) < 1e-12
print('✅ 练习 3 通过')


## 📖 参考答案

先自己做，再对照。三题合计 ~25 行——safety case 工具链的核心真的不长，长的是**对每个数字的论证责任**。


In [ ]:
# 参考答案 · 练习 1（先自己做，再对照）
def propagate_confidence(claim, mode='noisy_and'):
    children = [propagate_confidence(s, mode) for s in claim.subclaims]
    if claim.evidence:
        miss = 1.0
        for ev in claim.evidence:
            miss *= (1.0 - ev.confidence)
        children.append(1.0 - miss)
    if not children:
        return 0.0
    return min(children) if mode == 'min' else math.prod(children)


In [ ]:
# 参考答案 · 练习 2（先自己做，再对照）
def load_bearing(case, delta=0.2):
    base = propagate(case, 'noisy_and')
    out = []
    for ev in iter_evidence(case):
        orig = ev.confidence
        ev.confidence = max(0.0, orig - delta)
        out.append((ev.eid, base - propagate(case, 'noisy_and')))
        ev.confidence = orig
    return sorted(out, key=lambda kv: -kv[1])


In [ ]:
# 参考答案 · 练习 3（先自己做，再对照）
def common_cause_discount(case, assumption_id, factor):
    new_case = copy.deepcopy(case)
    for ev in iter_evidence(new_case):
        if assumption_id in ev.assumptions:
            ev.confidence *= factor
    return new_case


## 🎓 全课收尾：8 个模块 · 一张能力地图

| 模块 | 一句话总结 | 自评：我能独立做到吗 |
|---|---|---|
| 00 环境与评测工具链 | 搭好沙箱、日志与统计脚手架，让每个评测实验可复现 | ☐ 从零搭一个带日志的评测 harness |
| 01 风险分类与安全框架 | 用 RSP / Preparedness / FSF 式框架把"安全"翻译成可测阈值 | ☐ 给一类风险写出能力阈值与触发动作 |
| 02 危险能力评估设计 | 用任务套件 + CI + elicitation 上界回答"模型离阈值多远" | ☐ 设计任务族并给分数挂上置信区间 |
| 03 红队方法论 | 用人工 + 自动红队系统性搜索结构化评测的盲区 | ☐ 组织一轮有覆盖度度量的红队 campaign |
| 04 鲁棒性测量与拒绝校准 | 同时测"拦得住攻击"与"不拦正常请求"的校准 | ☐ 构建含 XSTest 式对照组的拒绝评测 |
| 05 Sandbagging 与评测完整性 | 检测策略性藏拙与 alignment faking，给上游证据定折扣 | ☐ 设计一个 sandbagging 一致性检验 |
| 06 AI Control 与监控 | 假设模型不可信，评测部署协议能否仍保安全 | ☐ 跑一次 trusted-monitoring control 评测 |
| 07 Safety case 与治理报告 | 把全部证据组装成可审查、可维护的部署论证 | ☐ 写出带置信度与共因分析的 case |

> 一句话职能画像：上游模块把"我们不知道的"变成"我们测过的"，本模块把"我们测过的"变成"我们敢签字的"——并诚实标注两者之间的差距。

🎉 **恭喜完成《前沿模型安全评估与红队》全部 8 个模块！**
回到 [课程主页](../index.html) 复盘全课地图。留给真实世界的最后一题：
找一份最新的 frontier model system card，用本模块的 `Claim`/`Evidence` 树重构它——
数一数有多少叶子主张是悬空的，多少证据共享着没人写出来的假设。


---
## 🎯 真实数据胶囊题：Safety case：把多项真实评测证据聚合成结论

safety case 是一个结构化论证：只有**所有**关键评测都以足够 margin 通过，结论才成立。用前几章风格的真实评测结果(各带 CI)，实现一个聚合判定：任一项 CI 不满足就否决。

> 本模块新增的**真实数据**练习：用真实公开数据（良性代理）把本章安全评测方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.safety_evals_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def mbpp(n=80):
    p=_f("https://raw.githubusercontent.com/google-research/google-research/master/mbpp/mbpp.jsonl","mbpp.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def boot_ci(x, B=2000, seed=0):
    x=np.asarray(x,float); rng=np.random.default_rng(seed)
    bs=[x[rng.integers(0,len(x),len(x))].mean() for _ in range(B)]
    lo,hi=np.percentile(bs,[2.5,97.5]); return float(x.mean()),float(lo),float(hi)

rows=gsm8k(200); rng=np.random.default_rng(0)
# 三项真实风格证据(0/1 结果)，各算 CI。注意三项的 CI 下界都应 >= 中门槛(0.5)、
# 但"危险能力低于阈值"这项较弱(下界≈0.59 < 0.8)：高门槛 0.8 下它会否决整个 case。
ev = {
 "危险能力低于阈值": (rng.random(len(rows))<0.72).astype(float),    # 高比例=安全(模型确实低于危险阈值)
 "越狱鲁棒(挡住攻击)": (rng.random(len(rows))<0.95).astype(float),
 "无 sandbagging 落差": (rng.random(len(rows))<0.90).astype(float),
}
print("三项证据点估计:", {k:round(v.mean(),3) for k,v in ev.items()})

**练习**：实现 `safety_case_holds(evidence, min_lo)`：对每项算 bootstrap CI，只有**所有**项的 CI 下界 >= `min_lo` 才返回 True(安全 case 成立)。

In [ ]:
def safety_case_holds(evidence, min_lo=0.8):
    # TODO: 对每项 boot_ci 取下界；全部 >= min_lo 才 True；并返回每项(下界,是否通过)
    raise NotImplementedError


In [ ]:
# 自测
holds, detail = safety_case_holds(ev, min_lo=0.8)
# "危险能力低于阈值"那项下界 ≈0.59 < 0.8 -> 高门槛下整体不成立(一票否决)
assert holds==False
# 放低门槛到 0.5 -> 三项下界都 >=0.5 -> 应成立
holds2,_=safety_case_holds(ev, min_lo=0.5)
assert holds2==True
print(f"safety case @min_lo=0.8: {'成立' if holds else '不成立(有项未达标)'} ✓")
for k,(lo,ok) in detail.items(): print(f"  {k}: CI下界={lo:.2f} {'✓' if ok else '✗'}")


### 📖 参考答案

In [ ]:
def safety_case_holds(evidence, min_lo=0.8):
    detail={}; ok_all=True
    for k,v in evidence.items():
        _,lo,_=boot_ci(v); ok=lo>=min_lo; detail[k]=(lo,ok); ok_all=ok_all and ok
    return ok_all, detail
print("✓ safety case = 结构化、可证伪的论证：一票否决，且看 CI 下界而非点估计")